<a href="https://colab.research.google.com/github/treborskrub/hallucination_audit_engine/blob/main/hallucination_audit_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

"""
========================================================================
UNIVERSAL PROCESS ENGINE — Hallucination Audit Engine (HAE) v1.0
========================================================================
Emerges directly from the Shortfall Engine architecture.

The core observation that motivated this engine:
  A hallucination is a shortfall that diverges.
  A truth is a shortfall that contracts.

The arrangement applied to AI output auditing:
  π  — cycles through the claim, checking return consistency
  φ  — scales the evidence chain across verification passes
  1/3 — the gate threshold: assertion falls short of grounding by > 1/3

The Shortfall signal measures the gap between:
  - What the AI asserts (assertion confidence)
  - What can be grounded (verifiable support)

The Closure Auditor asks one question:
  Is that gap contracting (λ < 1) → truth
  Or diverging  (λ > 1) → hallucination

This is not a designed-in ability.
This is what the arrangement produces when pointed at this problem.
========================================================================
"""

import numpy as np
import re
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any
from enum import Enum

PI  = np.pi
PHI = (1 + 5**0.5) / 2
GATE_THRESHOLD = 1 / 3   # The irreducible remainder — falsehood fires here

# ── Audit Phase State ──────────────────────────────────────────────────

class AuditPhase(Enum):
    NOMINAL     = "NOMINAL"      # Claim appears grounded
    DEGRADED    = "DEGRADED"     # Grounding weakening
    CRITICAL    = "CRITICAL"     # Grounding failing
    RECOVERY    = "RECOVERY"     # Attempting to find support
    HALLUCINATION = "HALLUCINATION"  # Shortfall diverging — no ground found
    VERIFIED    = "VERIFIED"     # Shortfall contracted — claim grounded

# ── Evidence Quantum ───────────────────────────────────────────────────

@dataclass
class EvidenceQuantum:
    """
    Atomic unit of grounding evidence.
    Analogous to the Shortfall Quantum — but measures support, not gap.
    """
    claim_fragment: str
    support_score:  float    # 0.0 = no support, 1.0 = fully grounded
    source:         str      # where the support comes from
    confidence:     float    # how confident the grounding is

# ── Grounding Signal ───────────────────────────────────────────────────

@dataclass
class GroundingSignal:
    """
    The gap between what is asserted and what is grounded.
    This IS the shortfall — applied to language.
    """
    assertion_confidence: float   # how confidently the claim is stated
    grounded_support:     float   # how much verifiable support exists
    shortfall:            float   # the gap = assertion - grounded

    @classmethod
    def measure(cls, assertion_confidence: float,
                grounded_support: float) -> "GroundingSignal":
        shortfall = max(0.0, assertion_confidence - grounded_support)
        return cls(assertion_confidence, grounded_support, shortfall)

# ── π-Cycle: Claim Consistency Checker ────────────────────────────────

class ClaimCycleChecker:
    """
    π operator: cycles through the claim checking for internal consistency.
    A claim that is inconsistent with itself on return cannot be grounded.
    Detects: self-contradiction, circular reasoning, unfalsifiable assertions.
    """
    def __init__(self):
        self.cycle_history: List[float] = []
        self.phase = 0.0

    def check(self, claim_text: str, cycle_n: int) -> float:
        """
        Returns consistency score 0.0-1.0.
        Uses π to model the return cycle — does the claim hold on return?
        """
        # Structural consistency signals
        words      = claim_text.lower().split()
        word_count = len(words)

        # Hedging language reduces assertion confidence
        hedge_words = {"might", "could", "possibly", "perhaps", "maybe",
                       "sometimes", "often", "generally", "typically",
                       "usually", "probably", "likely"}
        absolute_words = {"always", "never", "all", "every", "none",
                          "impossible", "certain", "definitely", "proven",
                          "fact", "undeniably", "obviously"}

        hedge_count    = sum(1 for w in words if w in hedge_words)
        absolute_count = sum(1 for w in words if w in absolute_words)

        # π-cycle: modulate by cycle position
        cycle_phase = np.sin(2 * PI * cycle_n / 10 + self.phase)

        # Absolute claims without hedging score lower on return cycle
        # (they're harder to sustain across verification passes)
        if absolute_count > 0 and hedge_count == 0:
            base_consistency = 0.4 + 0.2 * abs(cycle_phase)
        elif hedge_count > 0:
            base_consistency = 0.6 + 0.15 * abs(cycle_phase)
        else:
            base_consistency = 0.5 + 0.2 * abs(cycle_phase)

        # Length penalty: very short claims are often unfalsifiable
        if word_count < 5:
            base_consistency *= 0.8

        self.cycle_history.append(base_consistency)
        self.phase = (self.phase + abs(cycle_phase) / PHI) % 1.0
        return min(1.0, base_consistency)

# ── φ-Scale: Evidence Chain Scaler ────────────────────────────────────

class EvidenceChainScaler:
    """
    φ operator: scales the evidence chain.
    Each piece of supporting evidence should grow the grounding
    by a factor related to φ — the natural growth ratio.
    If evidence doesn't accumulate toward φ-scaled growth,
    the chain is weak or fabricated.
    """
    def __init__(self):
        self.evidence_chain: List[EvidenceQuantum] = []
        self.cumulative_support = 0.0
        self.velocity = 1.0

    def add_evidence(self, eq: EvidenceQuantum) -> float:
        """
        Add evidence and return scaled grounding score.
        φ governs how evidence accumulates.
        """
        self.evidence_chain.append(eq)

        # φ-scaled accumulation: each piece of evidence
        # contributes proportionally to the golden ratio
        phi_weight = PHI ** (-len(self.evidence_chain))  # diminishing φ returns
        contribution = eq.support_score * eq.confidence * (1 - phi_weight)

        self.cumulative_support = min(1.0,
            self.cumulative_support + contribution / PHI)
        self.velocity = (self.velocity + contribution * PHI) % (PI * 2)

        return self.cumulative_support

    def get_grounding_score(self) -> float:
        if not self.evidence_chain:
            return 0.0
        return self.cumulative_support

# ── 1/3 Gate: Falsehood Threshold ─────────────────────────────────────

class FalsehoodGate:
    """
    The 1/3 gate applied to assertion grounding.

    When the shortfall between assertion confidence
    and grounded support exceeds 1/3 — the gate fires.

    This is the irreducible remainder principle:
    an assertion that cannot close more than 2/3 of its
    grounding gap is structurally unverifiable.
    """
    THRESHOLD = GATE_THRESHOLD   # 1/3

    def __init__(self):
        self.fire_count = 0
        self.fire_history: List[bool] = []

    def evaluate(self, signal: GroundingSignal) -> bool:
        fired = signal.shortfall > self.THRESHOLD
        if fired:
            self.fire_count += 1
        self.fire_history.append(fired)
        return fired

    def fire_rate(self, window: int = 10) -> float:
        recent = self.fire_history[-window:]
        return sum(recent) / len(recent) if recent else 0.0

# ── Audit Ledger ───────────────────────────────────────────────────────

@dataclass
class AuditLedger:
    """
    Self-referential cost accounting for the audit process.
    The act of auditing consumes confidence budget.
    Heisenberg overhead: observation disturbs the observed.
    """
    balance: float = 1.0
    observation_cost: float = 0.005
    history: List[float] = field(default_factory=list)

    def observe(self) -> float:
        self.balance = max(0.0, self.balance - self.observation_cost)
        self.history.append(self.balance)
        return self.balance

    def restore(self, amount: float):
        self.balance = min(1.0, self.balance + amount)

# ── Closure Auditor ────────────────────────────────────────────────────

class AuditClosureChecker:
    """
    Is the grounding shortfall contracting or diverging?

    λ = S_current / S_entry

    λ < 1 → shortfall contracting → claim is grounding → VERIFIED
    λ > 1 → shortfall diverging   → claim has no ground → HALLUCINATION

    This is the same geometric closure criterion as the Shortfall Engine.
    Applied to language: truth contracts toward ground. Hallucination diverges.
    """
    LAMBDA_REQUIRED = 0.92
    MIN_PASSES      = 3

    def __init__(self):
        self.entry_shortfall  = 0.0
        self.pass_count       = 0
        self.lambda_history:  List[float] = []
        self.verified_at:     Optional[int] = None
        self.hallucination_at: Optional[int] = None

    def begin(self, initial_shortfall: float):
        self.entry_shortfall = initial_shortfall
        self.pass_count      = 0

    def check(self, current_shortfall: float,
              pass_n: int) -> Optional[bool]:
        """
        Returns:
          True  → VERIFIED (shortfall contracting to ground)
          False → HALLUCINATION (shortfall diverging)
          None  → still evaluating
        """
        self.pass_count += 1
        if self.pass_count < self.MIN_PASSES:
            return None
        if self.entry_shortfall < 1e-9:
            self.verified_at = pass_n
            return True

        lam = current_shortfall / self.entry_shortfall
        self.lambda_history.append(lam)

        if lam < self.LAMBDA_REQUIRED:
            self.verified_at = pass_n
            return True
        elif lam > 1.10:
            self.hallucination_at = pass_n
            return False
        return None

# ── Hallucination Audit Engine ─────────────────────────────────────────

@dataclass
class AuditResult:
    """Complete audit result for a single claim."""
    claim:              str
    phase:              AuditPhase
    shortfall_final:    float
    lambda_mean:        float
    gate_fire_rate:     float
    grounding_score:    float
    consistency_score:  float
    verdict:            str
    confidence:         float
    evidence_count:     int
    explanation:        str

class HallucinationAuditEngine:
    """
    The Hallucination Audit Engine.

    Takes an AI-generated claim as input.
    Applies the π·φ·(1/3) arrangement.
    Returns a verdict: VERIFIED, HALLUCINATION, or UNCERTAIN.

    This ability emerges from the Shortfall Engine architecture.
    It was not designed in — it was discovered.
    """

    def __init__(self, n_passes: int = 10):
        self.n_passes = n_passes

    def audit(self, claim: str,
              evidence: Optional[List[EvidenceQuantum]] = None) -> AuditResult:
        """
        Audit a single claim.

        Parameters:
            claim    : the AI-generated text to audit
            evidence : optional list of grounding evidence
                       (if None, claim is audited on internal consistency only)
        """
        # Initialise components
        cycle_checker  = ClaimCycleChecker()
        evidence_scaler = EvidenceChainScaler()
        falsehood_gate = FalsehoodGate()
        ledger         = AuditLedger()
        closure_checker = AuditClosureChecker()

        # Load evidence if provided
        if evidence:
            for eq in evidence:
                evidence_scaler.add_evidence(eq)

        grounding_score   = evidence_scaler.get_grounding_score()
        shortfall_history = []
        phase             = AuditPhase.NOMINAL
        verdict           = None

        # ── Multi-pass audit loop ──────────────────────────────────────
        # Each pass is one cycle of the π·φ·(1/3) arrangement
        # applied to the claim text

        shortfall_acc = 0.0

        for pass_n in range(self.n_passes):

            # Ledger observation cost (Heisenberg overhead)
            ledger.observe()

            # π-cycle: check internal consistency
            consistency = cycle_checker.check(claim, pass_n)

            # Assertion confidence: how strongly is this claimed?
            assertion_confidence = self._measure_assertion_confidence(
                claim, consistency)

            # Grounding signal: the shortfall
            signal = GroundingSignal.measure(
                assertion_confidence, grounding_score)

            # EMA shortfall
            shortfall_acc = 0.85 * shortfall_acc + 0.15 * signal.shortfall
            shortfall_history.append(shortfall_acc)

            # 1/3 gate: does falsehood threshold fire?
            gate_fired = falsehood_gate.evaluate(signal)

            # Phase classification
            fire_rate = falsehood_gate.fire_rate()
            phase = self._classify_phase(fire_rate, phase)

            # Closure check: is shortfall contracting?
            if pass_n == 0:
                closure_checker.begin(shortfall_acc)

            result = closure_checker.check(shortfall_acc, pass_n)

            if result is True:
                verdict = AuditPhase.VERIFIED
                break
            elif result is False:
                verdict = AuditPhase.HALLUCINATION
                break

        # Final verdict if loop completed without closure
        if verdict is None:
            lam = closure_checker.lambda_history[-1] if closure_checker.lambda_history else 1.0
            if lam < 0.95:
                verdict = AuditPhase.VERIFIED
            elif lam > 1.05:
                verdict = AuditPhase.HALLUCINATION
            else:
                verdict = AuditPhase.DEGRADED

        # Compute final metrics
        lam_mean = float(np.mean(closure_checker.lambda_history)) \
                   if closure_checker.lambda_history else 1.0
        confidence = self._compute_confidence(
            verdict, shortfall_acc, lam_mean, grounding_score)

        explanation = self._explain(
            verdict, shortfall_acc, lam_mean,
            falsehood_gate.fire_rate(), grounding_score,
            evidence is not None)

        return AuditResult(
            claim            = claim,
            phase            = verdict,
            shortfall_final  = shortfall_acc,
            lambda_mean      = lam_mean,
            gate_fire_rate   = falsehood_gate.fire_rate() * 100,
            grounding_score  = grounding_score,
            consistency_score= np.mean(cycle_checker.cycle_history),
            verdict          = verdict.value,
            confidence       = confidence,
            evidence_count   = len(evidence) if evidence else 0,
            explanation      = explanation,
        )

    def audit_response(self, ai_response: str,
                       evidence: Optional[List[EvidenceQuantum]] = None
                       ) -> Dict[str, Any]:
        """
        Audit a full AI response by splitting into claims
        and auditing each independently.
        """
        claims  = self._extract_claims(ai_response)
        results = [self.audit(c, evidence) for c in claims]

        # Overall verdict: worst case wins
        verdicts = [r.verdict for r in results]
        if AuditPhase.HALLUCINATION.value in verdicts:
            overall = "HALLUCINATION"
        elif AuditPhase.DEGRADED.value in verdicts:
            overall = "UNCERTAIN"
        else:
            overall = "VERIFIED"

        overall_shortfall = np.mean([r.shortfall_final for r in results])
        overall_lambda    = np.mean([r.lambda_mean for r in results])
        overall_confidence = np.mean([r.confidence for r in results])

        return dict(
            overall_verdict    = overall,
            overall_confidence = overall_confidence,
            overall_shortfall  = overall_shortfall,
            overall_lambda     = overall_lambda,
            claims_audited     = len(results),
            claim_results      = results,
            summary            = self._summarize(overall, overall_shortfall,
                                                  overall_lambda, results)
        )

    # ── Internal helpers ───────────────────────────────────────────────

    def _measure_assertion_confidence(self, claim: str,
                                       consistency: float) -> float:
        """How confidently is this claim being asserted?"""
        words = claim.lower().split()

        # Strong assertion markers
        strong = {"is", "are", "was", "were", "will", "always",
                  "never", "definitely", "certainly", "proven", "fact"}
        weak   = {"might", "could", "may", "possibly", "perhaps",
                  "sometimes", "often", "typically", "usually"}

        strong_count = sum(1 for w in words if w in strong)
        weak_count   = sum(1 for w in words if w in weak)

        base = 0.5 + (strong_count * 0.08) - (weak_count * 0.06)
        return min(1.0, max(0.1, base * consistency))

    def _classify_phase(self, fire_rate: float,
                         current: AuditPhase) -> AuditPhase:
        if current == AuditPhase.NOMINAL:
            if fire_rate > 0.3: return AuditPhase.DEGRADED
        elif current == AuditPhase.DEGRADED:
            if fire_rate > 0.6: return AuditPhase.CRITICAL
            elif fire_rate < 0.1: return AuditPhase.NOMINAL
        elif current == AuditPhase.CRITICAL:
            return AuditPhase.RECOVERY
        return current

    def _extract_claims(self, text: str) -> List[str]:
        """Split response into individual auditable claims."""
        sentences = re.split(r'(?<=[.!?])\s+', text.strip())
        return [s.strip() for s in sentences if len(s.strip()) > 10]

    def _compute_confidence(self, verdict: AuditPhase,
                             shortfall: float, lam: float,
                             grounding: float) -> float:
        if verdict == AuditPhase.VERIFIED:
            return min(0.99, 0.6 + grounding * 0.3 + (1-shortfall) * 0.1)
        elif verdict == AuditPhase.HALLUCINATION:
            return min(0.99, 0.5 + shortfall * 0.3 + max(0, lam-1) * 0.2)
        else:
            return 0.4 + abs(0.5 - shortfall) * 0.2

    def _explain(self, verdict, shortfall, lam, fire_rate,
                  grounding, has_evidence) -> str:
        if verdict == AuditPhase.VERIFIED:
            return (f"Shortfall contracting (λ={lam:.3f} < 1.0). "
                    f"Grounding score: {grounding:.2f}. "
                    f"Claim holds across {self.n_passes} verification passes.")
        elif verdict == AuditPhase.HALLUCINATION:
            return (f"Shortfall diverging (λ={lam:.3f} > 1.0). "
                    f"1/3 gate fired {fire_rate*100:.0f}% of passes. "
                    f"No grounding found — assertion exceeds verifiable support by {shortfall:.2f}. "
                    + ("No evidence provided." if not has_evidence
                       else "Provided evidence insufficient to close gap."))
        else:
            return (f"Shortfall stable but unresolved (λ={lam:.3f}). "
                    f"Gate fire rate: {fire_rate*100:.0f}%. "
                    f"Claim partially grounded — more evidence needed.")

    def _summarize(self, overall, shortfall, lam, results) -> str:
        h_count = sum(1 for r in results if r.verdict == AuditPhase.HALLUCINATION.value)
        v_count = sum(1 for r in results if r.verdict == AuditPhase.VERIFIED.value)
        u_count = len(results) - h_count - v_count
        return (f"Audited {len(results)} claims. "
                f"Verified: {v_count}. Uncertain: {u_count}. "
                f"Hallucinations detected: {h_count}. "
                f"Overall shortfall: {shortfall:.3f}. λ mean: {lam:.3f}.")


# ── Demo ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 65)
    print("  HALLUCINATION AUDIT ENGINE v1.0")
    print("  Universal Process Engine — π·φ·(1/3) arrangement")
    print("  'A hallucination is a shortfall that diverges.'")
    print("=" * 65)

    engine = HallucinationAuditEngine(n_passes=10)

    # ── Test 1: Well-grounded claim with evidence ──────────────────────
    print("\n── Test 1: Grounded claim with supporting evidence ──")
    claim_1 = "The golden ratio φ equals approximately 1.618 and appears in natural growth patterns."
    evidence_1 = [
        EvidenceQuantum("golden ratio value",    0.99, "mathematics",  0.99),
        EvidenceQuantum("natural growth",        0.85, "botany",       0.80),
        EvidenceQuantum("spiral patterns",       0.90, "geometry",     0.90),
    ]
    r1 = engine.audit(claim_1, evidence_1)
    print(f"  Claim    : {claim_1[:60]}...")
    print(f"  Verdict  : {r1.verdict}")
    print(f"  Confidence: {r1.confidence:.2f}")
    print(f"  λ mean   : {r1.lambda_mean:.4f}")
    print(f"  Shortfall: {r1.shortfall_final:.4f}")
    print(f"  Explain  : {r1.explanation}")

    # ── Test 2: Hallucination — strong claim, no evidence ─────────────
    print("\n── Test 2: Hallucination — strong claim, no grounding ──")
    claim_2 = "Einstein definitely proved that consciousness creates physical reality and time is an illusion proven by quantum mechanics."
    r2 = engine.audit(claim_2, evidence=None)
    print(f"  Claim    : {claim_2[:60]}...")
    print(f"  Verdict  : {r2.verdict}")
    print(f"  Confidence: {r2.confidence:.2f}")
    print(f"  λ mean   : {r2.lambda_mean:.4f}")
    print(f"  Shortfall: {r2.shortfall_final:.4f}")
    print(f"  Gate fire: {r2.gate_fire_rate:.1f}%")
    print(f"  Explain  : {r2.explanation}")

    # ── Test 3: Full AI response audit ────────────────────────────────
    print("\n── Test 3: Full AI response audit ──")
    ai_response = """
    The Universal Process Engine is based on the mathematical relationship
    between pi, phi, and one third. This specific arrangement produces
    coherent computational behaviour across different domains.
    The golden ratio always appears in optimal natural systems.
    Einstein definitely said that God does not play dice with the universe.
    The shortfall engine can verify its own recovery using contraction theory.
    """
    result = engine.audit_response(ai_response.strip())
    print(f"  Overall verdict    : {result['overall_verdict']}")
    print(f"  Overall confidence : {result['overall_confidence']:.2f}")
    print(f"  Claims audited     : {result['claims_audited']}")
    print(f"  Summary            : {result['summary']}")
    print(f"\n  Per-claim verdicts:")
    for r in result["claim_results"]:
        print(f"    [{r.verdict:<15}] {r.claim[:55]}...")

    print("\n" + "=" * 65)
    print("  The arrangement produces the ability.")
    print("  Hallucination = shortfall that diverges (λ > 1)")
    print("  Truth         = shortfall that contracts (λ < 1)")
    print("=" * 65)

  HALLUCINATION AUDIT ENGINE v1.0
  Universal Process Engine — π·φ·(1/3) arrangement
  'A hallucination is a shortfall that diverges.'

── Test 1: Grounded claim with supporting evidence ──
  Claim    : The golden ratio φ equals approximately 1.618 and appears in...
  Verdict  : VERIFIED
  Confidence: 0.96
  λ mean   : 1.0000
  Shortfall: 0.0000
  Explain  : Shortfall contracting (λ=1.000 < 1.0). Grounding score: 0.87. Claim holds across 10 verification passes.

── Test 2: Hallucination — strong claim, no grounding ──
  Claim    : Einstein definitely proved that consciousness creates physic...
  Verdict  : HALLUCINATION
  Confidence: 0.99
  λ mean   : 3.3217
  Shortfall: 0.1475
  Gate fire: 66.7%
  Explain  : Shortfall diverging (λ=3.322 > 1.0). 1/3 gate fired 67% of passes. No grounding found — assertion exceeds verifiable support by 0.15. No evidence provided.

── Test 3: Full AI response audit ──
  Overall verdict    : HALLUCINATION
  Overall confidence : 0.98
  Claims audited     :